In [18]:
import pandas as pd
file_path = './data/countries_affiliation_df_canada.csv'  # Replace with your actual file path
canada_df = pd.read_csv(file_path)

In [23]:
canada_df.iloc[0]["PMID"]


28350526

In [141]:
import os
import shutil
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
import time

class MyHandler(FileSystemEventHandler):
    def __init__(self, download_folder, pmid):
        self.download_folder = download_folder
        self.pmid = pmid
        self.downloaded_file_path = None

    def on_created(self, event):
        if not event.is_directory and event.src_path.endswith('.pdf'):
            self.downloaded_file_path = event.src_path
            print(f"New file detected: {event.src_path}")

    def get_downloaded_file_path(self):
        return self.downloaded_file_path

# Paths
chrome_driver_path = "/opt/homebrew/bin/chromedriver"  # Adjust this path to where you placed the chromedriver
download_folder = "/Users/yumcoder/Desktop/pubmed/download"  # Replace with your desired download folder path
# cookies_file = 'cookies.json'

def initialize_driver():
    service = Service(chrome_driver_path)
    options = webdriver.ChromeOptions()

    # Set the download folder for Chrome
    prefs = {
        "download.default_directory": download_folder,  # Change default directory for downloads
        "download.prompt_for_download": False,       # To auto download the file
        "plugins.always_open_pdf_externally": True,   # To disable PDF viewer and download the file directly
        "profile.default_content_setting_values.automatic_downloads": 1,
    }
    options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(service=service, options=options)

    # driver = webdriver.Safari()
    return driver

def login_and_save_cookies(driver):
    # Open the login page and perform login
    driver.get("https://pubmed-ncbi-nlm-nih-gov.myaccess.library.utoronto.ca/")
    
    # Wait for login elements and perform login (pseudo-code; adjust based on actual login process)
    wait = WebDriverWait(driver, 10)
    login_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "a.loginButton")))
    login_button.click()
     # Wait for the UTORid login page to appear
    WebDriverWait(driver, 10).until(EC.url_contains("utoronto.ca"))

    # Fill in the login form
    # Assuming you need to enter username and password and then click a login button
    # Adjust the locators and fields based on the actual UTORid login page structure
    username_field = wait.until(EC.presence_of_element_located((By.ID, "username")))
    password_field = wait.until(EC.presence_of_element_located((By.ID, "password")))
    submit_button = wait.until(EC.element_to_be_clickable((By.ID, "login-btn")))  # Adjust ID as needed
    username_field.send_keys("jafarin9")  # Replace with your actual username
    password_field.send_keys("=YumUofT2023!")  # Replace with your actual password
    submit_button.click()

    # Wait for the 'trust-browser-button' to appear and click it
    trust_browser_button = wait.until(EC.element_to_be_clickable((By.ID, "trust-browser-button")))
    trust_browser_button.click()
    # Wait for login to complete and save cookies
    WebDriverWait(driver, 30).until(EC.url_contains("https://pubmed-ncbi-nlm-nih-gov.myaccess.library.utoronto.ca/"))
    # id="trust-browser-button"

    # cookies = driver.get_cookies()
    # with open(cookies_file, 'w') as file:
    #     json.dump(cookies, file)
    # # print("Cookies saved.")

# def load_cookies(driver):
#     # Load cookies and add them to the browser
#     driver.get("https://pubmed-ncbi-nlm-nih-gov.myaccess.library.utoronto.ca/")
#     with open(cookies_file, 'r') as file:
#         cookies = json.load(file)
#     for cookie in cookies:
#         driver.add_cookie(cookie)

def download_pdfs(driver, pmid_list):
    # Loop through each PMID
    for index, row in pmid_list.iterrows():
        pmid = row['PMID']
        # if index > 20:
        #     break

        pdf_filename = os.path.join(download_folder, f"{pmid}.pdf")
        
        if os.path.exists(pdf_filename):
            print(f"File {pdf_filename} already exists, skipping...")
            continue

        try:
            # Open the PubMed article page
            driver.get(f"https://pubmed-ncbi-nlm-nih-gov.myaccess.library.utoronto.ca/{pmid}/")

            # Wait for the page to load and the full-text links to be visible
            wait = WebDriverWait(driver, 10)
            wait.until(EC.presence_of_element_located((By.CLASS_NAME, "full-text-links-list")))

            try:
                # Find all full-text links
                links = wait.until(EC.presence_of_all_elements_located((By.XPATH, "//div[@class='full-text-links-list']/a")))
                
                # Try each link until a successful download occurs
                download_successful = False
                for link in links:
                    link_text = link.get_attribute('textContent').strip()
                    link_href = link.get_attribute('href')
                    print(f"Trying link text: {link_text}")
                    print(f"Trying link href: {link_href}")
                    
                    # Click on the link
                    link.click()
                    
                    # Switch to the new tab if it opens
                    window_handles = driver.window_handles
                    if len(window_handles) > 1:
                        driver.switch_to.window(window_handles[-1])
                    else:
                        print("No new tab detected, continuing in the current tab.")


                    # Wait for the new page or current page to load the PDF link
                    wait.until(EC.presence_of_element_located((By.XPATH, "//a[text()='Download PDF'] | //*[contains(@href, '.pdf') or contains(@class, 'pdf') or contains(@data-article-pdf, 'pdf') or contains(@data-test, 'pdf') or contains(@data-download, 'pdf')]")))

                    # Attempt to find PDF link using multiple locators
                    pdf_links = driver.find_elements(By.XPATH, "//a[text()='Download PDF'] | //*[contains(@href, '.pdf') or contains(@class, 'pdf') or contains(@data-article-pdf, 'pdf') or contains(@data-test, 'pdf') or contains(@data-download, 'pdf')]")
                    # print('---pdf_links--->>>', [f.get_attribute("href") for f in pdf_links])
                    # Download the first valid PDF link found
                    for pdf_link in pdf_links:
                        pdf_url = pdf_link.get_attribute("href")
                       
                        if pdf_url and "www-sciencedirect-com" not in pdf_url.lower():
                            print(f"Found PDF link: {pdf_url}")
                           
                            if pdf_url.lower().endswith(".pdf"):
                                try:
                                    # Set up watchdog to monitor the download folder
                                    handler = MyHandler(download_folder, pmid)
                                    observer = Observer()
                                    observer.schedule(handler, download_folder, recursive=False)
                                    observer.start()
                                    print(f"Watching folder: {download_folder}")
                                    driver.get(pdf_url)

                                    # Wait for the file to appear and handle it
                                    while handler.get_downloaded_file_path() is None:
                                        time.sleep(1)
                                    
                                    downloaded_file_path = handler.get_downloaded_file_path()
                                    shutil.move(downloaded_file_path, pdf_filename)
                                    # download_button.click()
                                    # time.sleep(60)  # Wait for the download to complete
                                    download_successful = True
                                    break
                                finally:
                                    observer.stop()
                                    observer.join()


                            driver.get(pdf_url)  # Navigate to the PDF link to trigger download
                            driver.switch_to.window(driver.window_handles[-1])
                            #  | 
                            download_button = wait.until(EC.presence_of_element_located(
                                (By.XPATH, "//a[contains(@class, 'navbar-download') and contains(@href, 'download=true')]")
                            ))
                            if download_button:
                                # print("The element is a link.")
                                # Taylor & Francis
                                try:
                                    pdf_url_download_link = download_button.get_attribute("href")
                                    # Set up watchdog to monitor the download folder
                                    handler = MyHandler(download_folder, pmid)
                                    observer = Observer()
                                    observer.schedule(handler, download_folder, recursive=False)
                                    observer.start()
                                    print(f"Watching folder: {download_folder}")

                                    driver.get(pdf_url_download_link)
                                    driver.switch_to.window(driver.window_handles[-1])
                                    # Wait for the file to appear and handle it
                                    while handler.get_downloaded_file_path() is None:
                                        time.sleep(1)
                                    
                                    downloaded_file_path = handler.get_downloaded_file_path()
                                    shutil.move(downloaded_file_path, pdf_filename)
                                    # download_button.click()
                                    # time.sleep(60)  # Wait for the download to complete
                                    download_successful = True
                                    break
                                finally:
                                    driver.close()
                                    driver.switch_to.window(driver.window_handles[0])
                                    observer.stop()
                                    observer.join()

                    # Close the tab and switch back to the original tab if a new tab was opened
                    if len(driver.window_handles) > 1:
                        driver.close()
                        driver.switch_to.window(driver.window_handles[0])

                    if download_successful:
                        break  # Exit the link loop if successful

            except Exception as e:
                print(f"Full-text links not found for PMID: {pmid}: {e}")
        
        except Exception as e:
            print(f"An error occurred for PMID {pmid}: {e}")

def main():
    # Load PMIDs from the CSV file
    df_data = pd.read_csv('canada_pmid.csv', header=None, names=['PMID'])

    driver = initialize_driver()
    
    try:
        login_and_save_cookies(driver)

        # Perform the download
        download_pdfs(driver, df_data)

        time.sleep(600)
    finally:
        # Close the browser
        driver.quit()

if __name__ == "__main__":
    main()


SessionNotCreatedException: Message: session not created: Chrome failed to start: exited normally.
  (session not created: DevToolsActivePort file doesn't exist)
  (The process started from chrome location /Applications/Google Chrome.app/Contents/MacOS/Google Chrome is no longer running, so ChromeDriver is assuming that Chrome has crashed.)
Stacktrace:
0   chromedriver                        0x0000000102919024 cxxbridge1$str$ptr + 1887276
1   chromedriver                        0x0000000102911700 cxxbridge1$str$ptr + 1856264
2   chromedriver                        0x000000010252082c cxxbridge1$string$len + 88524
3   chromedriver                        0x0000000102551ad0 cxxbridge1$string$len + 289904
4   chromedriver                        0x000000010254d890 cxxbridge1$string$len + 272944
5   chromedriver                        0x000000010258c82c cxxbridge1$string$len + 530892
6   chromedriver                        0x0000000102559474 cxxbridge1$string$len + 321044
7   chromedriver    

In [144]:
! pip install webdriver-manager
 

275552.27s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl (27 kB)


In [41]:
! pip install pyperclip

  Preparing metadata (setup.py) ... done
  Created wheel for pyperclip: filename=pyperclip-1.9.0-py3-none-any.whl size=11002 sha256=2bae6a68fbdfea549a91b2cff024fb1e61085f4d274aa6f298b027b75764830f
  Stored in directory: /Users/yumcoder/Library/Caches/pip/wheels/e0/e8/fc/8ab8aa326e33bc066ccd5f3ca9646eab4299881af933f94f09
Successfully built pyperclip


In [43]:
from selenium import webdriver
import pyperclip
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from selenium.webdriver.common.keys import Keys
import os

def initialize_driver():
    """
    Initialize the Chrome WebDriver with specified options and service.
    """
    options = Options()
    # options.add_argument("--headless")  # Ensure GUI is off
    # options.add_argument("--no-sandbox")
    # options.add_argument("--disable-dev-shm-usage")

    service = Service("/opt/homebrew/bin/chromedriver")  # Adjust the path if needed
    driver = webdriver.Chrome(service=service, options=options)
    return driver

def fill_textarea(driver, wait, title, pmid):
    """
    Fill the last textarea found on the page with the specified text.
    """
    try:
        # Wait until the textarea is clickable
        textarea = wait.until(EC.element_to_be_clickable((By.XPATH, "//textarea[last()]")))
        textarea.clear()
        # Part 1
        textarea.send_keys("Given the provided academic paper:")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)  # New line
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)  # Another new line
        textarea.send_keys(f"- **{title}**")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)  # New line

        textarea.send_keys(f"- **PMID (PubMed)**: {pmid}")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)  # New line
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)  # Another new line
        
        textarea.send_keys("extract and return the following information in JSON format:")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)  # New line

        # Part 2
        textarea.send_keys("1. **Patient Level Data:** A boolean value indicating whether the paper works on patient-level data.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        
        textarea.send_keys("2. **Institutions Involved:** A list of institutions that participated in the data collection process.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        
        textarea.send_keys("3. **Countries:** A list of countries associated with the institutions that participated in data collection.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        
        textarea.send_keys("4. **Data Collection Approach:** A string indicating whether the data was collected in a central repository (\"central\") or in a distributed manner (\"distributed\").")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        
        textarea.send_keys("5. **Total Dataset Size:** The exact count of individual records or samples.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)

        textarea.send_keys("6. **pmid:** PubMed id.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        
        # Part 3
        textarea.send_keys("**Expected/Example of JSON format:**")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        
        textarea.send_keys("```json")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        
        textarea.send_keys("{")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys('  "patient_level_data": true,')
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys('  "institutions_involved": ["Institution A", "Institution B"],')
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys('  "countries": ["Country X", "Country Y"],')
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys('  "data_collection_approach": "central",')
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys('  "total_dataset_size": 1')
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys('  "pmid": 28589480')
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("}")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("```")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        
        # Part 4
        textarea.send_keys("Please analyze the paper and return ONLY the JSON object accordingly without any additional text; I mean only JSON should be in the result.")
        

        print("Text filled in the textarea.")
        textarea.send_keys(Keys.ENTER) #submit
    except Exception as e:
        print(f"Error filling textarea: {e}")

def click_copy_button(driver, wait):
    """
    Scroll to the button with a clipboard icon and click it to copy the response.
    """
    try:
        # Wait for the SVG element
        clipboard_svg = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "svg[data-icon='clipboard']")))

        # Get the parent button
        copy_button = clipboard_svg.find_element(By.XPATH, './ancestor::button')
        #print('copy_button>>>', copy_button)

        # Scroll the element into view
        driver.execute_script("arguments[0].scrollIntoView(true);", copy_button)
        #driver.save_screenshot('before_click.png')

        # Wait until the button is clickable
        # copy_button.click()
        driver.execute_script("arguments[0].click();", copy_button)

        print("Copy button clicked.")
    except Exception as e:
        print(f"Error clicking copy button: {e}")

def main():
    driver = initialize_driver()
    wait = WebDriverWait(driver, 10)  # Wait up to 10 seconds

    # data = [
    #     {
    #         "title": "Preliminary testing by adults of a haptics-assisted robot platform designed for children with physical impairments to access play",
    #         "pmid": 28696831
    #     },
    #     {
    #         "title": "Dark Flow, Depression and Multiline Slot Machine Play",
    #         "pmid": 28589480
    #     }
    # ]

    try:
        file_path = './data/countries_affiliation_df_canada.csv'  # Replace with your actual file path
        canada_df = pd.read_csv(file_path)
        print(f"len:{len(canada_df)}")
        driver.get("https://www.perplexity.ai/")
        original_window = driver.current_window_handle  # Store the handle of the original window
        input("Press any key to continue...")

        # for entry in data:
        for index, entry in canada_df.iterrows():
            # body = driver.find_element(By.TAG_NAME, 'body')
            # body.send_keys(Keys.COMMAND + 'k')  # For macOS

            # title = entry["title"]
            # pmid = entry["pmid"]
            title = entry["ArticleTitle"]
            pmid = entry["PMID"]

            # Check if file already exists
            filename = f"./perplexity/{pmid}.txt"
            if os.path.exists(filename):
                print(f"File {filename} already exists. Skipping...")
                continue  # Skip to the next iteration

            # Open a new tab and switch to it
            driver.execute_script("window.open('');")
            new_tab = driver.window_handles[-1]
            driver.switch_to.window(new_tab)

            # Load the page in the new tab
            driver.get("https://www.perplexity.ai/")

            fill_textarea(driver, wait, title, pmid)

            click_copy_button(driver, wait)

            # Allow time for the clipboard to update
            time.sleep(1)

            # Retrieve the string from the clipboard and print it
            clipboard_content = pyperclip.paste()
            print(f"Clipboard Content for PMID {pmid}:\n", clipboard_content)

            # Save the result to a file
            with open(filename, "w") as file:
                file.write(clipboard_content)

            # Close the new tab and switch back to the original tab
            driver.close()
            driver.switch_to.window(original_window)

    finally:
        # Ensure the driver is properly closed even if an error occurs
        driver.quit()

if __name__ == "__main__":
    main()


len:7756
File ./perplexity/28350526.txt already exists. Skipping...
File ./perplexity/28444633.txt already exists. Skipping...
File ./perplexity/28550657.txt already exists. Skipping...
File ./perplexity/28589480.txt already exists. Skipping...
File ./perplexity/28596271.txt already exists. Skipping...
File ./perplexity/28601499.txt already exists. Skipping...
File ./perplexity/28651909.txt already exists. Skipping...
File ./perplexity/28673115.txt already exists. Skipping...
File ./perplexity/28687962.txt already exists. Skipping...
File ./perplexity/28696831.txt already exists. Skipping...
File ./perplexity/28712346.txt already exists. Skipping...
File ./perplexity/28722148.txt already exists. Skipping...
File ./perplexity/28726811.txt already exists. Skipping...
File ./perplexity/28756487.txt already exists. Skipping...
File ./perplexity/28771699.txt already exists. Skipping...
File ./perplexity/28836083.txt already exists. Skipping...
File ./perplexity/28836836.txt already exists. 

KeyboardInterrupt: 

In [44]:
import os
import json

def extract_json_objects_from_text(text):
    json_objects = []
    while True:
        # Find the first '{' and the last '}'
        start_index = text.find('{')
        end_index = text.rfind('}')

        if start_index == -1 or end_index == -1 or start_index > end_index:
            break

        # Extract the substring from the first '{' to the last '}'
        json_str = text[start_index:end_index + 1]

        # Try to parse the extracted substring as JSON
        try:
            json_object = json.loads(json_str)
            json_objects.append(json_object)
            # Remove the processed part of the text
            text = text[end_index + 1:]
        except json.JSONDecodeError:
            # If parsing fails, continue searching in the remaining text
            text = text[end_index + 1:]

    return json_objects

def extract_json_objects_from_txt(file_path):
    with open(file_path, 'r') as file:
        content = file.read()
    return extract_json_objects_from_text(content)

def read_json_from_folder(folder_path):
    all_json_objects = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.txt'):
            file_path = os.path.join(folder_path, file_name)
            json_objects = extract_json_objects_from_txt(file_path)
            all_json_objects.extend(json_objects)
    return all_json_objects

# Example usage
folder_path = './perplexity/'
json_array = read_json_from_folder(folder_path)
print(json_array)


[{'patient_level_data': False, 'institutions_involved': [], 'countries': [], 'data_collection_approach': 'distributed', 'total_dataset_size': 0, 'pmid': 29926728}, {'patient_level_data': False, 'institutions_involved': [], 'countries': [], 'data_collection_approach': 'distributed', 'total_dataset_size': 0, 'pmid': 29790333}, {'patient_level_data': True, 'institutions_involved': ['Department of Psychiatry'], 'countries': ['Not specified'], 'data_collection_approach': 'distributed', 'total_dataset_size': 183, 'pmid': 29454222}, {'patient_level_data': True, 'institutions_involved': ['Institution A', 'Institution B'], 'countries': ['Country X', 'Country Y'], 'data_collection_approach': 'central', 'total_dataset_size': 5090, 'pmid': 29704323}, {'patient_level_data': False, 'institutions_involved': [], 'countries': [], 'data_collection_approach': 'central', 'total_dataset_size': 0, 'pmid': 29671385}, {'patient_level_data': False, 'institutions_involved': [], 'countries': [], 'data_collection

In [45]:
df_perplexity = pd.DataFrame(json_array)
df_perplexity

,patient_level_data,institutions_involved,countries,data_collection_approach,total_dataset_size,pmid
0,False,[],[],distributed,0.0,29926728
1,False,[],[],distributed,0.0,29790333
2,True,[Department of Psychiatry],[Not specified],distributed,183.0,29454222
3,True,"[Institution A, Institution B]","[Country X, Country Y]",central,5090.0,29704323
4,False,[],[],central,0.0,29671385
...,...,...,...,...,...,...
389,True,[],[],central,116.0,28756487
390,False,"[Beef Cattle Institute, Kansas State University]",[United States],distributed,0.0,29385611
391,True,"[Nizhny Novgorod State Medical Academy, Instit...",[Russia],central,1024.0,28853237
392,True,"[University of Calgary, University of Alberta,...",[Canada],distributed,57.0,30155514


In [46]:
df_perplexity[(df_perplexity['patient_level_data'] == True) & (df_perplexity['data_collection_approach'] != 'central')]

,patient_level_data,institutions_involved,countries,data_collection_approach,total_dataset_size,pmid
2,True,[Department of Psychiatry],[Not specified],distributed,183.0,29454222
16,True,"[Agriculture and Agri-Food Canada, University ...",[Canada],distributed,190.0,30149509
17,True,"[University of Calgary, University of Alberta,...",[Canada],distributed,57.0,30155514
31,True,"[Institution A, Institution B]","[Country X, Country Y]",distributed,27.0,29244902
52,True,[Department of Psychiatry],[Not specified],distributed,183.0,29454222
...,...,...,...,...,...,...
382,True,[Centre Hospitalier Regional et Universitaire ...,"[France, Canada]",distributed,1000.0,30154742
383,True,"[Brigham and Women’s Hospital, Harvard Medical...","[USA, Portugal, Canada, Japan]",distributed,9.0,29869321
384,True,"[Institution A, Institution B]","[Country X, Country Y]",distributed,48.0,29344313
388,True,[],[],distributed,4.0,29382744


In [1]:
import pandas as pd
file_path = 'data-full-text.csv'  # Replace with your actual file path
canada_df = pd.read_csv(file_path)

In [2]:
canada_df

,pid,title
0,29609039,Computer-aided diagnosis of cavernous malforma...
1,29616521,The relationship between cerebral oxygen satur...
2,29632364,Multimodal and Multiscale Deep Neural Networks...
3,29636016,Machine learning classification of first-episo...
4,29642656,Socioeconomic disparities and difficulties to ...
...,...,...
3524,39001096,Derivative Method to Detect Sleep and Awake St...
3525,39001410,Deep Learning Algorithms for Bladder Cancer Se...
3526,39002043,Predictors of High Healthcare Cost Among Patie...
3527,39012781,Short-Term Exposure to Wildfire-Specific PM2.5...


In [25]:
from selenium import webdriver
import pyperclip
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from selenium.webdriver.common.keys import Keys
import os

def initialize_driver():
    """
    Initialize the Chrome WebDriver with specified options and service.
    """
    options = Options()
    # options.add_argument("--headless")  # Ensure GUI is off
    # options.add_argument("--no-sandbox")
    # options.add_argument("--disable-dev-shm-usage")
 
    # Enable clipboard access
    options.add_experimental_option("prefs", {
        "profile.default_content_setting_values.clipboard": 1  # 1: allow, 2: block
    })

    service = Service("/opt/homebrew/bin/chromedriver")  # Adjust the path if needed
    driver = webdriver.Chrome(service=service, options=options)
    return driver

def fill_textarea(driver, wait, title, pmid):
    """
    Fill the last textarea found on the page with the specified text.
    """
    try:
        # Wait until the textarea is clickable
        textarea = wait.until(EC.element_to_be_clickable((By.XPATH, "//textarea[last()]")))
        textarea.clear()
        # Part 1
        textarea.send_keys("Consider this paper I MEAN FULL TEXT OF THE PAPER,")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)  # New line
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)  # Another new line
        textarea.send_keys(f"- **{title}**")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)  # New line

        textarea.send_keys(f"- **PMID (PubMed)**: {pmid}")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)  # New line
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)  # Another new line
        
        textarea.send_keys("extract and return the following information in JSON format:")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)  # New line

        # Part 2
        textarea.send_keys("1. **Total Dataset Size:**")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Number of Records/Samples: Exact count of individual records or samples.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Size in GB/TB: The physical size of the dataset.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Number of Data Points per Sample: Average number of data points collected per sample.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)

        textarea.send_keys("2. **Data Collection Method:**")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Type: Centralized vs. Decentralized.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Source of Data: Specific institutions, countries, or regions where data was collected.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Timeframe of Data Collection: Period during which the data was collected.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)

        textarea.send_keys("3. **Data Modalities:**")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Type of Data: Imaging, genomic, clinical, sensor, etc.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Combination of Modalities: Whether multiple data types were used together (e.g., multimodal learning).")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Data Format: Structured, unstructured, semi-structured.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)

        textarea.send_keys("4. **Number of Features in Datasets:**")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Feature Count: Exact number of features.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Feature Types: Categorical, numerical, text, image, etc.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Feature Selection Methods: Techniques used to select or engineer features.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)

        textarea.send_keys("5. **AI Methods:**")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Specific Algorithms: Deep learning, support vector machines, random forests, etc.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Model Architecture: Details on model design, such as layers in neural networks.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Training Techniques: Techniques like transfer learning, ensemble learning, federated learning.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Performance Metrics: Accuracy, precision, recall, F1 score, AUC, etc.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)

        textarea.send_keys("6. **Number of Collaborators:**")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Number of Institutions: Count of participating organizations.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Geographical Distribution: Countries or regions involved.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Roles: Specific roles of collaborators (e.g., data providers, algorithm developers, model trainers).")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)

        textarea.send_keys("7. **Data Privacy Techniques:**")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Privacy-Preserving Methods: Differential privacy, homomorphic encryption, secure multi-party computation.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Compliance with Regulations: GDPR, HIPAA, etc.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Data Anonymization: Techniques for anonymizing sensitive information.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)

        textarea.send_keys("8. **Data Heterogeneity:**")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Source Diversity: Variability in data sources (e.g., hospitals, clinics, sensors).")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Population Diversity: Diversity of patient demographics (e.g., age, ethnicity, gender).")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Consistency Across Datasets: Measures taken to standardize data for analysis.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)

        textarea.send_keys("9. **Data Sharing and Collaboration Mechanisms:**")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Platforms Used: Specific platforms or frameworks (e.g., GA4GH, DataSHIELD).")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Collaboration Models: Types of collaboration (e.g., open, consortium-based).")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("   - Data Accessibility: Level of access (open access, restricted access, etc.).")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)

        textarea.send_keys("10. **Ethical Considerations:**")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("    - Ethical Approvals: Details on ethics committee approvals.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("    - Informed Consent: How informed consent was obtained from participants.")
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys("    - Bias Mitigation: Steps taken to address bias in data collection and analysis.")
        
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        textarea.send_keys(Keys.SHIFT + Keys.ENTER)
        
        # Part 4
        textarea.send_keys("Please analyze the paper and return ONLY the JSON object accordingly without any additional text; I mean only JSON should be in the result.")
        

        print("Text filled in the textarea.")
        textarea.send_keys(Keys.ENTER) #submit
    except Exception as e:
        print(f"Error filling textarea: {e}")

def click_copy_button(driver, wait):
    """
    Scroll to the button with a clipboard icon and click it to copy the response.
    """
    try:
        # Wait for the SVG element
        clipboard_svg = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "svg[data-icon='clipboard']")))

        # Get the parent button
        copy_button = clipboard_svg.find_element(By.XPATH, './ancestor::button')
        #print('copy_button>>>', copy_button)

        # Scroll the element into view
        driver.execute_script("arguments[0].scrollIntoView(true);", copy_button)
        #driver.save_screenshot('before_click.png')

        # Wait until the button is clickable
        # copy_button.click()
        driver.execute_script("arguments[0].click();", copy_button)

        print("Copy button clicked.")
    except Exception as e:
        print(f"Error clicking copy button: {e}")

def main():
    driver = initialize_driver()
    wait = WebDriverWait(driver, 10)  # Wait up to 10 seconds

    # data = [
    #     {
    #         "title": "Preliminary testing by adults of a haptics-assisted robot platform designed for children with physical impairments to access play",
    #         "pmid": 28696831
    #     },
    #     {
    #         "title": "Dark Flow, Depression and Multiline Slot Machine Play",
    #         "pmid": 28589480
    #     }
    # ]

    try:
        file_path = 'data-full-text.csv'  # Replace with your actual file path
        canada_df = pd.read_csv(file_path)
        print(f"len:{len(canada_df)}")
        driver.get("https://www.perplexity.ai/")
        original_window = driver.current_window_handle  # Store the handle of the original window
        input("Press any key to continue...")

        # for entry in data:
        for index, entry in canada_df.iterrows():
            pyperclip.copy('')  # Clear the clipboard before starting
            # body = driver.find_element(By.TAG_NAME, 'body')
            # body.send_keys(Keys.COMMAND + 'k')  # For macOS

            # if index> 1:
            #     break

            # title = entry["title"]
            # pmid = entry["pmid"]
            title = entry["title"]
            pmid = entry["pid"]

            # Check if file already exists
            filename = f"./perplexity-ft/{pmid}.txt"
            if os.path.exists(filename):
                print(f"File {filename} already exists. Skipping...")
                continue  # Skip to the next iteration

            # Open a new tab and switch to it
            driver.execute_script("window.open('');")
            new_tab = driver.window_handles[-1]
            driver.switch_to.window(new_tab)

            # Load the page in the new tab
            driver.get("https://www.perplexity.ai/")

            fill_textarea(driver, wait, title, pmid)

            click_copy_button(driver, wait)

            # Allow time for the clipboard to update
            time.sleep(1)

            # Retrieve the string from the clipboard and print it
            clipboard_content = pyperclip.paste()
            print(f"Clipboard Content for PMID {pmid}:\n", clipboard_content)

            # Save the result to a file
            with open(filename, "w") as file:
                file.write(clipboard_content)

            # Close the new tab and switch back to the original tab
            # input("Press Enter to close the browser...")
            driver.close()
            driver.switch_to.window(original_window)
    finally:
        # Ensure the driver is properly closed even if an error occurs
        driver.quit()

if __name__ == "__main__":
    main()


len:3529
File ./perplexity-ft/29609039.txt already exists. Skipping...
File ./perplexity-ft/29616521.txt already exists. Skipping...
File ./perplexity-ft/29632364.txt already exists. Skipping...
File ./perplexity-ft/29636016.txt already exists. Skipping...
File ./perplexity-ft/29642656.txt already exists. Skipping...
File ./perplexity-ft/29708944.txt already exists. Skipping...
File ./perplexity-ft/29854259.txt already exists. Skipping...
File ./perplexity-ft/29860189.txt already exists. Skipping...
File ./perplexity-ft/30290728.txt already exists. Skipping...
File ./perplexity-ft/30300751.txt already exists. Skipping...
File ./perplexity-ft/30301895.txt already exists. Skipping...
File ./perplexity-ft/33934542.txt already exists. Skipping...
File ./perplexity-ft/30992063.txt already exists. Skipping...
File ./perplexity-ft/32193665.txt already exists. Skipping...
File ./perplexity-ft/32864270.txt already exists. Skipping...
File ./perplexity-ft/32864596.txt already exists. Skipping...

In [17]:
# import os
# import hashlib

# def hash_file(filepath):
#     """Generate a hash for the contents of a file."""
#     hasher = hashlib.md5()
#     with open(filepath, 'rb') as file:
#         buf = file.read()
#         hasher.update(buf)
#     return hasher.hexdigest()

# def find_and_delete_duplicates(folder_path):
#     """Find and delete original text files if duplicates exist in the specified folder."""
#     file_hashes = {}
#     duplicates_to_delete = []

#     # First pass: Identify duplicates
#     for filename in os.listdir(folder_path):
#         if filename.endswith(".txt"):
#             file_path = os.path.join(folder_path, filename)
#             file_hash = hash_file(file_path)

#             if file_hash in file_hashes:
#                 # Add both the original and the duplicate to the deletion list
#                 duplicates_to_delete.append(file_path)
#                 if file_hashes[file_hash] is not None:
#                     duplicates_to_delete.append(file_hashes[file_hash])
#                     file_hashes[file_hash] = None  # Mark original for deletion
#             else:
#                 file_hashes[file_hash] = file_path

#     # Second pass: Delete files
#     for file_path in set(duplicates_to_delete):  # Use set to avoid double deletion
#         os.remove(file_path)
#         print(f"Deleted file: {file_path}")

# if __name__ == "__main__":
#     folder_path = "./perplexity-ft/"  # Replace with your folder path
#     find_and_delete_duplicates(folder_path)


Deleted file: ./perplexity-ft/29194610.txt
Deleted file: ./perplexity-ft/32356095.txt
Deleted file: ./perplexity-ft/33113221.txt
Deleted file: ./perplexity-ft/31414853.txt
Deleted file: ./perplexity-ft/31782551.txt
Deleted file: ./perplexity-ft/32561782.txt
Deleted file: ./perplexity-ft/30018571.txt
Deleted file: ./perplexity-ft/37658409.txt
Deleted file: ./perplexity-ft/31104700.txt
Deleted file: ./perplexity-ft/38265444.txt
Deleted file: ./perplexity-ft/33131680.txt
Deleted file: ./perplexity-ft/29237237.txt
Deleted file: ./perplexity-ft/31179487.txt
Deleted file: ./perplexity-ft/31284154.txt
Deleted file: ./perplexity-ft/32670113.txt
Deleted file: ./perplexity-ft/30573458.txt
Deleted file: ./perplexity-ft/33958739.txt
Deleted file: ./perplexity-ft/32318076.txt
Deleted file: ./perplexity-ft/32563597.txt
Deleted file: ./perplexity-ft/32715024.txt
Deleted file: ./perplexity-ft/32803448.txt
Deleted file: ./perplexity-ft/31133782.txt
Deleted file: ./perplexity-ft/29717941.txt
Deleted fil

In [26]:
import os
import json

def extract_json_objects_from_text(text):
    json_objects = []
    while True:
        # Find the first '{' and the last '}'
        start_index = text.find('{')
        end_index = text.rfind('}')

        if start_index == -1 or end_index == -1 or start_index > end_index:
            break

        # Extract the substring from the first '{' to the last '}'
        json_str = text[start_index:end_index + 1]

        # Try to parse the extracted substring as JSON
        try:
            json_object = json.loads(json_str)
            json_objects.append(json_object)
            # Remove the processed part of the text
            text = text[end_index + 1:]
        except json.JSONDecodeError:
            # If parsing fails, continue searching in the remaining text
            text = text[end_index + 1:]

    return json_objects

def extract_json_objects_from_txt(file_path):
    with open(file_path, 'r') as file:
        content = file.read()
    return extract_json_objects_from_text(content)

def read_json_from_folder(folder_path):
    all_json_objects = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.txt'):
            file_path = os.path.join(folder_path, file_name)
            json_objects = extract_json_objects_from_txt(file_path)
            # If no JSON objects were found, add an empty object
            if not json_objects:
                json_objects.append({})
            # Add the JSON objects with the corresponding PID (file name without extension)
            for json_object in json_objects:
                pid = os.path.splitext(file_name)[0]
                all_json_objects.append({'pid': pid, 'data': json_object})
    return all_json_objects

# Example usage
folder_path = './perplexity-ft/'
json_array = read_json_from_folder(folder_path)
print(json_array)


[{'pid': '30659193', 'data': {'Total Dataset Size': {'Number of Records/Samples': 174, 'Size in GB/TB': 'Not specified', 'Number of Data Points per Sample': 'Not specified'}, 'Data Collection Method': {'Type': 'Centralized', 'Source of Data': 'Singapore and Taiwan', 'Timeframe of Data Collection': 'Not specified'}, 'Data Modalities': {'Type of Data': 'Neuroimaging (fMRI)', 'Combination of Modalities': 'Single modality', 'Data Format': 'Structured'}, 'Number of Features in Datasets': {'Feature Count': 'Not specified', 'Feature Types': 'Not specified', 'Feature Selection Methods': 'Not specified'}, 'AI Methods': {'Specific Algorithms': 'Ensemble learning', 'Model Architecture': 'Stacked predictions from multiple models', 'Training Techniques': 'Not specified', 'Performance Metrics': {'Accuracy': '87%', 'Other Metrics': 'Not specified'}}, 'Number of Collaborators': {'Number of Institutions': 5, 'Geographical Distribution': ['Singapore', 'Taiwan'], 'Roles': ['Data providers', 'Algorithm de

In [29]:
json_array

[{'pid': '30659193',
  'data': {'Total Dataset Size': {'Number of Records/Samples': 174,
    'Size in GB/TB': 'Not specified',
    'Number of Data Points per Sample': 'Not specified'},
   'Data Collection Method': {'Type': 'Centralized',
    'Source of Data': 'Singapore and Taiwan',
    'Timeframe of Data Collection': 'Not specified'},
   'Data Modalities': {'Type of Data': 'Neuroimaging (fMRI)',
    'Combination of Modalities': 'Single modality',
    'Data Format': 'Structured'},
   'Number of Features in Datasets': {'Feature Count': 'Not specified',
    'Feature Types': 'Not specified',
    'Feature Selection Methods': 'Not specified'},
   'AI Methods': {'Specific Algorithms': 'Ensemble learning',
    'Model Architecture': 'Stacked predictions from multiple models',
    'Training Techniques': 'Not specified',
    'Performance Metrics': {'Accuracy': '87%',
     'Other Metrics': 'Not specified'}},
   'Number of Collaborators': {'Number of Institutions': 5,
    'Geographical Distributio

In [30]:
import json
import psycopg2

# Connect to PostgreSQL
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="yum",
    host="localhost",
    port="5432"
)
cursor = conn.cursor()

# Insert data into the table
for record in json_array:
    cursor.execute(
        """
        INSERT INTO perplexity (pid, data)
        VALUES (%s, %s::jsonb)
        """,
         (record['pid'], json.dumps(record['data']))
    )

# Commit the transaction and close the connection
conn.commit()
cursor.close()
conn.close()


In [35]:
! pip install pandas scikit-learn nltk


In [33]:
file_path = 'data-full-text.csv'  # Replace with your actual file path
canada_df = pd.read_csv(file_path)
canada_df

,pid,title
0,29609039,Computer-aided diagnosis of cavernous malforma...
1,29616521,The relationship between cerebral oxygen satur...
2,29632364,Multimodal and Multiscale Deep Neural Networks...
3,29636016,Machine learning classification of first-episo...
4,29642656,Socioeconomic disparities and difficulties to ...
...,...,...
3524,39001096,Derivative Method to Detect Sleep and Awake St...
3525,39001410,Deep Learning Algorithms for Bladder Cancer Se...
3526,39002043,Predictors of High Healthcare Cost Among Patie...
3527,39012781,Short-Term Exposure to Wildfire-Specific PM2.5...


In [38]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import nltk
from nltk.corpus import stopwords

# Download stopwords
nltk.download('stopwords')

# Load your data
data = canada_df

# Assuming your dataset has a 'text' column that contains the content to analyze
texts = data['title'].fillna('')  # Fill NaNs with empty strings

# Preprocess the text
stop_words = stopwords.words('english')  # Use a list instead of a set
vectorizer = TfidfVectorizer(stop_words=stop_words)
X = vectorizer.fit_transform(texts)

# Fit the KMeans model
num_clusters = 15  # Change this based on your needs
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
kmeans.fit(X)

# Assign cluster labels back to the original DataFrame
data['theme'] = kmeans.labels_



[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/yumcoder/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
import pandas as pd
file_path = './data-modalities-usa.csv'  # Replace with your actual file path
data = pd.read_csv(file_path)
data

,pid,modalities,modalities2,x
0,28952798,{Medical},"{Blood,Pressure,Systolic,BP,Diastolic,BP}","{Blood,BP,Diastolic,Medical,Pressure,Systolic}"
1,28968847,{MRI},{MRI},{MRI}
2,28981875,{Nil},"{Text,Audio}","{Audio,Nil,Text}"
3,29194610,"{dental-related,claims}","{topical,fluoride,dental,sealants}","{claims,dental,dental-related,fluoride,sealant..."
4,29251172,"{Gene,expression,profile}","{gene,expression}","{expression,gene,Gene,profile}"
...,...,...,...,...
904,31926806,"{""null"",""null"",""null"",""null"",""null""}",NaN,"{""null""}"
905,38783089,"{cervical,extension,traction,(CET)}",NaN,"{cervical,(CET),extension,traction}"
906,37829182,"{Medical,Imaging,Electronic,Health,Records}",NaN,"{Electronic,Health,Imaging,Medical,Records}"
907,36280681,"{Ultrasound,Clinical}",NaN,"{Clinical,Ultrasound}"


In [54]:
import pandas as pd
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import re

stemmer = PorterStemmer()

def normalize_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove non-alphanumeric characters except for whitespace
    text = re.sub(r'[^\w\s]', '', text)
    # Tokenize and stem
    tokens = word_tokenize(text)
    normalized_tokens = [stemmer.stem(token) for token in tokens]
    return ' '.join(normalized_tokens)

def process_names(names):
    if pd.isna(names):
        return []
    # Normalize and split
    names_list = [normalize_text(name.strip()) for name in names.replace('""', '"').split(',')]
    # Remove duplicates and return as a list
    return list(set(names_list))

# Apply the function to the 'x' column
data['unique_names'] = data['x'].apply(process_names)

# Display the DataFrame with the new column
data.head(5)


,pid,countries,x,modalities,modalities2,unique_names
0,28601499,"[""Australia"", ""Canada""]","[""fMRI""], [""fMRI""]","[""fMRI""]","[""fMRI""]",[fmri]
1,28651909,"[""Canada"", ""China""]","[""Computerized Cambridge Neurocognitive Test A...","[""Computerized Cambridge Neurocognitive Test A...","[""cognitive tests""]",[immedi and delay logic memori of the wechsler...
2,28756487,"[""Canada"", ""France"", ""United Kingdom""]","[""anatomical"", ""functional""], [""voice-sensitiv...","[""anatomical"", ""functional""]","[""voice-sensitive cortex"", ""functional voice a...","[anatom, function, function voic activ, voices..."
3,28771699,"[""Brazil"", ""Canada"", ""Israel""]","[""Checkerboard DNA-DNA hybridisation""], [""DNA-...","[""Checkerboard DNA-DNA hybridisation""]","[""DNA-DNA hybridisation""]","[dnadna hybridis, checkerboard dnadna hybridis]"
4,28853237,"[""Canada"", ""Russia""]","[""Optical Coherence Tomography""], [""optical co...","[""Optical Coherence Tomography""]","[""optical coherence tomography""]",[optic coher tomographi]


In [16]:
# for names in data.head(5)['unique_names'].dropna():
#     print(names)  # Update the set with unique names
all_modalities = [modality for sublist in data['unique_names'].dropna() for modality in sublist]
all_modalities

['systol',
 'medic',
 'bp',
 'blood',
 'diastol',
 'pressur',
 'mri',
 'text',
 'nil',
 'audio',
 'fluorid',
 'claim',
 'dentalrel',
 'dental',
 'sealant',
 'topic',
 'gene',
 'express',
 'profil',
 'scale',
 'likert',
 'selfreport',
 'xray',
 'comput',
 'miccai',
 'tomographi',
 'mri',
 'scan',
 'magnet',
 'reson',
 'imag',
 'structur',
 't2weight',
 'mri',
 'weight',
 'diffusionweight',
 't1weight',
 't2',
 'genet',
 'data',
 'sampl',
 'blood',
 'hematolog',
 'mood',
 'energi',
 'physiolog',
 'cancer',
 'tomographi',
 'comput',
 'therapi',
 'prostat',
 'radiat',
 'treatment',
 'mri',
 'magnet',
 'reson',
 'imag',
 'function',
 'audio',
 'methyl',
 'imag',
 'text',
 'dna',
 'vision',
 'feedback',
 'kinesthet',
 'modal',
 'intent',
 'profil',
 'brain',
 'transcript',
 'gene',
 'express',
 'medic',
 'text',
 'imag',
 'freedom',
 'knee',
 'degre',
 '6',
 'robot',
 'of',
 'system',
 'intern',
 'kinemat',
 'tibial',
 'test',
 'rotat',
 'type',
 'microrna',
 'fluid',
 'cerebrospin',
 'level

In [46]:
output_file_path = 'data-modalities-updated.csv'
data.to_csv(output_file_path, index=False)

In [55]:
import pandas as pd
from collections import Counter

# Load the updated data
# file_path = 'data-modalities-updated.csv'
# data = pd.read_csv(file_path)

# Flatten the list of modalities
# all_modalities = [modality for sublist in data['unique_names'].dropna() for modality in eval(sublist)]
all_modalities = [modality for sublist in data['unique_names'].dropna() for modality in sublist]
# Count frequencies
modality_counts = Counter(all_modalities)

# Find the top 10 most common modalities
top_10_modalities = modality_counts.most_common(50)

# Convert to DataFrame for better readability
top_10_df = pd.DataFrame(top_10_modalities, columns=['Modality', 'Count'])

# Display the top 10 modalities
print(top_10_df)

# Optionally, save the results to a CSV file
top_10_file_path = 'top_10_modalities.csv'
top_10_df.to_csv(top_10_file_path, index=False)

print(f"Top 10 modalities saved to {top_10_file_path}")


                      Modality  Count
0                         imag    309
1                         text    266
2                          mri    205
3                       clinic     98
4                           ct     72
5                   ultrasound     67
6            comput tomographi     66
7                          eeg     61
8            magnet reson imag     60
9                                  55
10                  medic imag     51
11                        fmri     46
12                       video     46
13                       genom     41
14                     ct scan     40
15                         ecg     36
16                       audio     35
17                 clinic data     31
18                machin learn     28
19                         pet     26
20      electron health record     26
21        comput tomographi ct     26
22                       medic     25
23                gene express     25
24                   demograph     25
25          

In [50]:
top_10_file_path


'top_10_modalities.csv'

In [22]:
import pandas as pd

# Data USA and Canada (modalities)
modalities_data = {
    'Modality': ['MRI', 'CT Scans', 'Genomics', 'fMRI', 'Ultrasound', 'EEG'],
    'Papers': [87, 56, 33, 21, 19, 19],
}

# Total number of papers
total_papers = 909

# Calculate percentage
modalities_data['Percentage'] = [(x / total_papers) * 100 for x in modalities_data['Papers']]

# Create DataFrame
df = pd.DataFrame(modalities_data)

df


,Modality,Papers,Percentage
0,MRI,87,9.570957
1,CT Scans,56,6.160616
2,Genomics,33,3.630363
3,fMRI,21,2.310231
4,Ultrasound,19,2.090209
5,EEG,19,2.090209


In [23]:
import pandas as pd
file_path = './data-uk.csv'  # Replace with your actual file path
data = pd.read_csv(file_path)
data

,pid,countries,x,modalities,modalities2
0,28756487,"[""Canada"", ""France"", ""United Kingdom""]","[""anatomical"", ""functional""], [""voice-sensitiv...","[""anatomical"", ""functional""]","[""voice-sensitive cortex"", ""functional voice a..."
1,29043544,"[""Canada"", ""Poland"", ""United Kingdom""]","[""Transcranial Doppler (TCD)"", ""Intracranial p...","[""Transcranial Doppler (TCD)"", ""Intracranial p...","[""Transcranial Doppler"", ""Intracranial pressur..."
2,29274736,"[""Canada"", ""Mexico"", ""United Kingdom""]","[""Tissue Density Estimates"", ""Cortical Thickne...","[""Tissue Density Estimates"", ""Cortical Thickne...","[""magnetic resonance images""]"
3,29335008,"[""Australia"", ""Canada"", ""United Kingdom""]","[""16S"", ""Shotgun metagenomics""], [""16S rRNA ge...","[""16S"", ""Shotgun metagenomics""]","[""16S rRNA gene"", ""shotgun metagenomics""]"
4,29340580,"[""Canada"", ""Denmark"", ""Russia"", ""South Korea"",...","[""Blood Samples""], [""Hematological""]","[""Blood Samples""]","[""Hematological""]"
...,...,...,...,...,...
256,37829182,"[""Australia"", ""Canada"", ""China"", ""Colombia"", ""...",NaN,"[""Medical Imaging"", ""Electronic Health Records""]",NaN
257,34694234,"[""Canada"", ""United Kingdom"", ""United States""]",NaN,"[""Clinical""]",NaN
258,31919503,"[""Canada"", ""Turkey"", ""United Kingdom"", ""United...",NaN,NaN,NaN
259,31926806,"[""Australia"", ""Brazil"", ""Canada"", ""Germany"", ""...",NaN,"[null, null, null, null, null]",NaN


In [27]:
# New data for Canada and UK
modalities_data_uk = {
    'Modality': ['MRI', 'Genomics', 'EEG', 'CT Scans', 'fMRI'],
    'Papers': [24, 16, 11, 9, 5],
}

# Total number of papers for Canada and UK
total_papers_uk = 267

# Calculate percentage
modalities_data_uk['Percentage'] = [(x / total_papers_uk) * 100 for x in modalities_data_uk['Papers']]

# Create DataFrame
df_uk = pd.DataFrame(modalities_data_uk)

df_uk


,Modality,Papers,Percentage
0,MRI,24,8.988764
1,Genomics,16,5.992509
2,EEG,11,4.119850
3,CT Scans,9,3.370787
4,fMRI,5,1.872659


In [36]:
import pandas as pd
file_path = './data-china.csv'  # Replace with your actual file path
data = pd.read_csv(file_path)
data

,pid,countries,x,modalities,modalities2
0,28651909,"[""Canada"", ""China""]","[""Computerized Cambridge Neurocognitive Test A...","[""Computerized Cambridge Neurocognitive Test A...","[""cognitive tests""]"
1,28869436,"[""Canada"", ""China""]","[""Lower Limbs Rehabilitation Robot""], [""Robot""...","[""Lower Limbs Rehabilitation Robot""]","[""Robot"", ""Device""]"
2,29098645,"[""Canada"", ""China"", ""Netherlands""]","[""Cognitive features""], [""Cognitive features""]","[""Cognitive features""]","[""Cognitive features""]"
3,29268169,"[""Canada"", ""China"", ""France"", ""Poland"", ""Unite...","[""MICCAI""], [""X-ray"", ""Computed Tomography""]","[""MICCAI""]","[""X-ray"", ""Computed Tomography""]"
4,29356346,"[""Canada"", ""China""]","[""Audiogram"", ""Initial Hearing Level"", ""Time E...","[""Audiogram"", ""Initial Hearing Level"", ""Time E...","[""Audiogram"", ""Hearing level""]"
...,...,...,...,...,...
232,38842257,"[""Canada"", ""China"", ""Germany"", ""Israel"", ""Neth...","[""Serum metabolites"", ""Magnetic resonance ente...","[""Serum metabolites"", ""Magnetic resonance ente...","[""Serum metabolites""]"
233,38850215,"[""Canada"", ""China"", ""Japan""]","[""MRI""], [""Structural connectivity data""]","[""MRI""]","[""Structural connectivity data""]"
234,38951965,"[""Canada"", ""China""]","[""Medical""], [""Image"", ""Text""]","[""Medical""]","[""Image"", ""Text""]"
235,39013848,"[""Australia"", ""Canada"", ""China"", ""Germany"", ""I...",NaN,"[""Magnetic Resonance"", ""Brain Imaging""]",NaN


In [34]:
# Data provided
data = {
    'Technology': ['MRI', 'CT scans', 'fMRI', 'Genomics', 'EEG'],
    'Count': [29, 10, 7, 6, 5]
}

# Creating a DataFrame
import pandas as pd

df = pd.DataFrame(data)

# Calculate the total count
total_count = 237  # Given total

# Calculate percentages
df['Percentage'] = (df['Count'] / total_count) * 100

# Display the DataFrame with percentages
print(df)


  Technology  Count  Percentage
0        MRI     29   12.236287
1   CT scans     10    4.219409
2       fMRI      7    2.953586
3   Genomics      6    2.531646
4        EEG      5    2.109705


In [44]:
import pandas as pd
file_path = './data-ger.csv'  # Replace with your actual file path
data = pd.read_csv(file_path)
data

,pid,countries,x,modalities,modalities2
0,28968847,"[""Canada"", ""Germany"", ""United States""]","[""MRI""], [""MRI""]","[""MRI""]","[""MRI""]"
1,28981875,"[""Canada"", ""Germany"", ""Turkey"", ""United States""]","[""Nil""], [""Text"", ""Audio""]","[""Nil""]","[""Text"", ""Audio""]"
2,29272464,"[""Canada"", ""Czechia"", ""Germany"", ""Turkey"", ""Un...","[""Structural magnetic resonance imaging scans ...","[""Structural magnetic resonance imaging scans ...","[""MRI scans""]"
3,29454222,"[""Canada"", ""Czechia"", ""Germany"", ""Turkey""]","[""Magnetic Resonance Imaging"", ""Lipids and bod...","[""Magnetic Resonance Imaging"", ""Lipids and bod...","[""structural MRI"", ""fasting lipids"", ""body mas..."
4,29539639,"[""Australia"", ""Austria"", ""Canada"", ""Denmark"", ...","[""DNA methylation""], [""text"", ""image"", ""audio""]","[""DNA methylation""]","[""text"", ""image"", ""audio""]"
...,...,...,...,...,...
204,38842593,"[""Canada"", ""Germany""]","[""Metabolomics""], [""Metabolomic"", ""Xenobiotic""]","[""Metabolomics""]","[""Metabolomic"", ""Xenobiotic""]"
205,38848770,"[""Canada"", ""Germany""]","[""Computed Tomography""], [""Image data""]","[""Computed Tomography""]","[""Image data""]"
206,38926357,"[""Canada"", ""Germany""]","[""Histopathology images"", ""Whole genome sequen...","[""Histopathology images"", ""Whole genome sequen...","[""Histopathology images""]"
207,39013848,"[""Australia"", ""Canada"", ""China"", ""Germany"", ""I...",NaN,"[""Magnetic Resonance"", ""Brain Imaging""]",NaN


In [52]:
# Data provided
data = {
    'Technology': ['MRI', 'CT scans', 'fMRI', 'PET', 'EEG'],
    'Count': [35, 16, 23, 6, 9]
}
# Creating a DataFrame
import pandas as pd

df = pd.DataFrame(data)

# Calculate the total count
total_count = 209  # Given total

# Calculate percentages
df['Percentage'] = (df['Count'] / total_count) * 100

# Display the DataFrame with percentages
print(df)


  Technology  Count  Percentage
0        MRI     35   16.746411
1   CT scans     16    7.655502
2       fMRI     23   11.004785
3        PET      6    2.870813
4        EEG      9    4.306220


In [53]:
import pandas as pd
file_path = './data-ca.csv'  # Replace with your actual file path
data = pd.read_csv(file_path)
data

,pid,countries,x,modalities,modalities2
0,28601499,"[""Australia"", ""Canada""]","[""fMRI""], [""fMRI""]","[""fMRI""]","[""fMRI""]"
1,28651909,"[""Canada"", ""China""]","[""Computerized Cambridge Neurocognitive Test A...","[""Computerized Cambridge Neurocognitive Test A...","[""cognitive tests""]"
2,28756487,"[""Canada"", ""France"", ""United Kingdom""]","[""anatomical"", ""functional""], [""voice-sensitiv...","[""anatomical"", ""functional""]","[""voice-sensitive cortex"", ""functional voice a..."
3,28771699,"[""Brazil"", ""Canada"", ""Israel""]","[""Checkerboard DNA-DNA hybridisation""], [""DNA-...","[""Checkerboard DNA-DNA hybridisation""]","[""DNA-DNA hybridisation""]"
4,28853237,"[""Canada"", ""Russia""]","[""Optical Coherence Tomography""], [""optical co...","[""Optical Coherence Tomography""]","[""optical coherence tomography""]"
...,...,...,...,...,...
2678,37829182,"[""Australia"", ""Canada"", ""China"", ""Colombia"", ""...",NaN,"[""Medical Imaging"", ""Electronic Health Records""]",NaN
2679,36280681,"[""Canada"", ""Saudi Arabia"", ""United States""]",NaN,"[""Ultrasound"", ""Clinical""]",NaN
2680,31919503,"[""Canada"", ""Turkey"", ""United Kingdom"", ""United...",NaN,NaN,NaN
2681,38290583,"[""Brazil"", ""Canada"", ""Norway""]",NaN,NaN,NaN


In [56]:
# Data provided
data = {
    'Technology': ['MRI', 'CT scans', 'Ultrasound', 'EEG', 'Genomics'],
    'Count': [363, 177, 88, 77, 69]
}

# Creating a DataFrame
import pandas as pd

df = pd.DataFrame(data)

# Calculate the total count
total_count = 2683  # Given total

# Calculate percentages
df['Percentage'] = (df['Count'] / total_count) * 100

# Display the DataFrame with percentages
print(df)


   Technology  Count  Percentage
0         MRI    363   13.529631
1    CT scans    177    6.597093
2  Ultrasound     88    3.279911
3         EEG     77    2.869922
4    Genomics     69    2.571748


In [57]:
! pip install matplotlib seaborn pandas
